In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 9.15/1000, lbd = (0.147 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 306.6

shape = np.sin((np.pi*days)/365)                       

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 1.319464841628453
Average value is: 0.8400000000000005


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 972.6
AI = PET/RPY
print(AI)

Total rainfall over 450 years = 493.881 meters
Average rainfall per year = 493.881 mm/year
1.9693008904538412


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                  

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 36.0   # LOU sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_DI = 0.3625919956120978
TI_tex = 0.7068222210935676
TI Total = 0.5347071083528327


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.8   #m/d,
kstot = 1     #m/d
phitex = 0.43  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [18:05<00:00, 8069.11it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       879.859657   91.068056   54.194709  0.059376  0.001743  0.219554   
2       625.441219   46.984704   64.161876  0.043261  0.001048  0.113275   
3       579.190408   24.490508   70.925488  0.042355  0.000774  0.059044   
4       562.785675   12.978471   83.372959  0.038474  0.000486  0.031289   
5       553.686352    7.054881   95.734369  0.036482  0.000299  0.017008   
...            ...         ...         ...       ...       ...       ...   
996    1813.152181  168.141170  215.919052  0.119112  0.003874  0.405338   
997    1813.166524  172.846474  201.082067  0.139182  0.004542  0.416679   
998    1813.155461  175.313293  203.315628  0.125533  0.004104  0.422628   
999    1813.084786  172.820534  216.154626  0.120420  0.003930  0.416618   
1000   1813.043235  173.995863  211.740395  0.128094  0.004184  0.419450   

           

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Lou\LOU.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\LOU.csv", index= True, header = True))